# TMJDLC — Javtiful Telegram Bot (Google Colab)

Run the admin-only Pyrogram bot from this notebook. It uses Neon PostgreSQL for the durable queue and settings, downloads Javtiful videos at fixed 720p, and uploads videos/screenshots to your configured Telegram channels.

**Keep this Colab runtime running** while the bot is processing jobs. A runtime reset stops the bot and clears local downloaded media, but queued jobs and channel settings remain in Neon.


## 1. Add your credentials to Colab Secrets

Open the **key icon (Secrets)** in the Colab left sidebar and add these names. Enable notebook access for each secret:

- `TG_API_ID`
- `TG_API_HASH`
- `TG_BOT_TOKEN`
- `TG_ADMIN_IDS` — your Telegram numeric user ID (multiple IDs can be comma-separated)
- `DATABASE_URL` — Neon connection string for the `Javtiful-uploader` database

Do not paste credentials into notebook cells or commit them to GitHub. The notebook reads them from Colab Secrets and does not print them.


In [ ]:
#@title 2. Pull latest main and install TMJDLC
%cd /content
!if [ -d /content/TMJDLC/.git ]; then git -C /content/TMJDLC pull --ff-only origin main; else git clone --depth 1 --branch main https://github.com/Telebotfaroff/TMJDLC.git /content/TMJDLC; fi
%cd /content/TMJDLC
!python -m pip install --upgrade pip
!python -m pip install -e .
!sudo apt-get update -qq && sudo apt-get install -y ffmpeg
!python -m playwright install --with-deps chromium
!python -m compileall -q tmjdlc
import subprocess
TMJDLC_COMMIT = subprocess.check_output(['git', '-C', '/content/TMJDLC', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('TMJDLC checkout:', TMJDLC_COMMIT)
print('Bot module compiled successfully.')


In [ ]:
#@title 3. Load Colab Secrets and configure the worker
import os
from google.colab import userdata

SECRET_NAMES = ['TG_API_ID', 'TG_API_HASH', 'TG_BOT_TOKEN', 'TG_ADMIN_IDS', 'DATABASE_URL']
missing = []
for key in SECRET_NAMES:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()
if missing:
    raise RuntimeError('Add these missing Colab Secrets, enable notebook access, then rerun this cell: ' + ', '.join(missing))

#@markdown Worker tuning (1–4 workers). Keep this at 2 for a balanced Colab setup.
MAX_PARALLEL = 2 #@param {type:'slider', min:1, max:4, step:1}
JOB_MAX_ATTEMPTS = 3 #@param {type:'integer'}
JOB_LEASE_SECONDS = 3600 #@param {type:'integer'}
QUEUE_POLL_SECONDS = 4 #@param {type:'integer'}
os.environ['TMJDLC_MAX_PARALLEL'] = str(MAX_PARALLEL)
os.environ['TMJDLC_JOB_MAX_ATTEMPTS'] = str(JOB_MAX_ATTEMPTS)
os.environ['TMJDLC_JOB_LEASE_SECONDS'] = str(JOB_LEASE_SECONDS)
os.environ['TMJDLC_QUEUE_POLL_SECONDS'] = str(QUEUE_POLL_SECONDS)
os.environ['TMJDLC_WORK_DIR'] = '/content/tmjdlc-bot'
os.makedirs(os.environ['TMJDLC_WORK_DIR'], exist_ok=True)
print('Secrets loaded (values hidden).')
print(f'Worker configuration: {MAX_PARALLEL} parallel workers; retry limit {JOB_MAX_ATTEMPTS}.')


In [ ]:
#@title 4. Verify Neon connection and initialize the queue
from tmjdlc.bot_queue import JobQueue
queue = JobQueue(max_attempts=JOB_MAX_ATTEMPTS)
queue.initialize()
counts = queue.counts()
print('Neon connection: OK')
print('Queue tables: ready')
print('Existing job counts:', counts or 'No jobs yet')
print('Destination channels are stored in Neon. Use /settings in Telegram to view them, or /setup to configure them.')


## 5. Start the Telegram bot

Run the next cell and leave it running. The bot starts its Pyrogram connection, recovers expired queue jobs, and launches the configured workers. Use the Telegram bot privately from an ID listed in `TG_ADMIN_IDS`.

First-time Telegram setup: send `/setup` to the bot, then provide your video destination channel and screenshot destination channel (or send `SKIP`). The bot must be an administrator with posting permission in both destination channels.


In [ ]:
#@title 6. Run TMJDLC (keep this cell running)
%cd /content/TMJDLC
!python -u -m tmjdlc.bot


## Telegram commands and channel downloader

- Send a Javtiful channel URL such as `https://javtiful.com/channel/madonna` to inspect it.
- Choose **All pages**, **Page range**, or **Select pages**.
- Advanced options let you toggle screenshots, skip existing jobs, and adjust parallel workers. Quality is fixed at **720p**; Javtiful pages are treated as 23 videos per page, with the last page counted from its actual listing.
- `/setup` — configure video and screenshot destinations.
- `/settings` — show saved destinations and worker settings.
- `/dl POST_URL` — queue one video.
- `/queue` — show queue counts and recent jobs.
- `/cancel JOB_ID` — cancel a queued job.
- `/retry JOB_ID` — retry a failed job.

The Neon database stores queue jobs and channel IDs. Downloaded files are stored under `/content/tmjdlc-bot` and are temporary to the Colab runtime. If Colab resets, rerun setup cells and start the bot again; Neon queue history remains available.
